In [ ]:
# Construye el modelo
import pyomo.environ as pe

# Resuelve el modelo
import pyomo.opt as po

In [ ]:
model = pe.ConcreteModel()

In [ ]:
print("hola")

Sets

In [ ]:
model.product_types = pe.Set(initialize = ["A","B","C"])
model.factories = pe.Set(initialize = ["F1","F2","F3","F4","F5","F6",])

Parameters

In [ ]:
capacity_dict = {
    "F1":550,
    "F2":700,
    "F3":1100,
    "F4":350,
    "F5":400,
    "F6":450
}
model.capacity_of_factory = pe.Param( model.factories, initialize=capacity_dict)


In [ ]:
cost_dict = {
    ("A", "F1"): 25, ("A", "F2"): 30, ("A", "F3"): 26, ("A", "F4"): 34, ("A", "F5"): 32, ("A", "F6"): 30,
    ("B", "F1"): 30, ("B", "F2"): 32, ("B", "F3"): 34, ("B", "F4"): 35, ("B", "F5"): 38, ("B", "F6"): 40,
    ("C", "F1"): 40, ("C", "F2"): 46, ("C", "F3"): 42, ("C", "F4"): 37, ("C", "F5"): 40, ("C", "F6"): 50
}

model.unitary_cost = pe.Param(model.product_types, model.factories, initialize=cost_dict)

In [ ]:
sold_dict = {
    "A": 700,
    "B": 500,
    "C": 600
}

model.already_sold = pe.Param(model.product_types, initialize=sold_dict)

In [ ]:
price_dict = {
    "A": 60,
    "B": 82.5,
    "C": 108
}

model.selling_price = pe.Param(model.product_types, initialize=price_dict)

Variables

In [ ]:
model.product_quantity = pe.Var(model.product_types, model.factories, within = pe.NonNegativeReals)

Objective Function

In [ ]:
def obj_rule(model):
    revenue = sum(model.selling_price[p] * model.product_quantity[p, f]
                  for p in model.product_types for f in model.factories)
    cost = sum(model.unitary_cost[p, f] * model.product_quantity[p, f]
               for p in model.product_types for f in model.factories)
    return revenue - cost

model.profit = pe.Objective(rule=obj_rule, sense=pe.maximize)

Constraints

In [ ]:
def contract_rule(model, p):
    return sum(model.product_quantity[p,f] for f in model.factories) >= model.already_sold[p]

model.contract_constraint = pe.Constraint(model.product_types, rule=contract_rule)

In [ ]:
def capacity_rule(model, f):
    return sum(model.product_quantity[p,f] for p in model.product_types) <= model.capacity_of_factory[f]

model.capacity_constraint = pe.Constraint(model.factories, rule=capacity_rule)

In [ ]:
#solver = po.SolverFactory('glpk')
solver = po.SolverFactory('gurobi', executable = r"C:\gurobi1303\win64\bin\gurobi_cl.exe")
results = solver.solve(model, tee=True) 

Set parameter Username
Set parameter LicenseID to value 2860157
Set parameter LogFile to value "gurobi.log"
Using license file C:\Users\Alonso\gurobi.lic
Academic license - for non-commercial use only - expires 2027-09-03

Usage: gurobi_cl [--command]* [param=value]* filename
Type 'gurobi_cl --help' for more information.


In [ ]:
print(pe.value(model.profit))


ERROR: evaluating object as numeric value: product_quantity[A,F1]
        (object: <class 'pyomo.core.base.var.VarData'>)
    No value for uninitialized VarData object product_quantity[A,F1]
ERROR: evaluating object as numeric value: profit
        (object: <class 'pyomo.core.base.objective.ScalarObjective'>)
    No value for uninitialized VarData object product_quantity[A,F1]


ValueError: No value for uninitialized VarData object product_quantity[A,F1]

In [ ]:
solver = po.SolverFactory("glpk")
results = solver.solve(model)

print("Status:", results.solver.status)
print("Termination:", results.solver.termination_condition)
print("Profit:", pe.value(model.profit))

solver 'glpk'


ApplicationError: No executable found for solver 'glpk'